#### Get data from Kaggle

In [ ]:
!kaggle datasets download -d dhanushnarayananr/credit-card-fraud
!unzip credit-card-fraud.zip
!rm -rf credit-card-fraud.zip
!mv card_transdata.csv ../../data/card_transdata.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

##### feature info

distance_from_home: continous

distance_from_last_transaction: continous

ratio_to_median_purchase_price: continous

repeat_retailer: binary 

used_chip: binary

used_pin_number: binary 

online_order: binary

__fraud: binary (target)__

In [ ]:
data = pd.read_csv('../../data/card_transdata.csv')
print(data.columns)
data.hist()
data.describe()

# Standardizing

Standardizing continous features - distances and ratios have very different ranges

In [ ]:
from sklearn.preprocessing import StandardScaler

continuous_features = ['distance_from_home', 'distance_from_last_transaction', 'ratio_to_median_purchase_price']
scaler = StandardScaler()
data[continuous_features] = scaler.fit_transform(data[continuous_features])

# Balancing

https://en.wikipedia.org/wiki/Oversampling_and_undersampling_in_data_analysis

Should do some balancing on the binary features - use [SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html) in imblearn library

SMOTENC has been the best (results wise)

In [ ]:
X = data.drop(columns=['fraud'])
y = data['fraud']

X.describe()

In [ ]:
X.hist()

In [ ]:
from imblearn.over_sampling import SMOTE, ADASYN, SMOTENC

# this takes a lot longer
# X, y = SMOTENC(categorical_features=[3,4,5,6]).fit_resample(X, y)
X, y = ADASYN().fit_resample(X, y)
# y.hist()

In [ ]:
X.describe()

In [ ]:
X.hist()

# Models

### Basic logistic regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

logit_model = LogisticRegression()
logit_model.fit(X_train, y_train)
y_preds = logit_model.predict(X_test)

In [ ]:
for pred, actual in zip(y_test[:20], y_preds[:20]):
    print(f'{actual} : {pred}')

Metrics

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import confusion_matrix

print(f'Accuracy: {logit_model.score(X_test, y_test)}')
print(f'Precision: {precision_score(y_test, y_preds)}')
print(f'F1 score: {f1_score(y_test, y_preds)}')
print(f'Recall: {recall_score(y_test, y_preds)}')
print(f'Confusion Matrix: \n{confusion_matrix(y_test, y_preds)}')

In [ ]:
# now I want to see the regression coefficients
dict(zip(logit_model.coef_.flatten(), X.columns))

These coefficients look really solid:

- Holding all else constant, for every unit increase in `ratio_to_median_purchase_price` we expect an increase of __2.99__ in likelihood that the purchase was __fraudulent__.
- Holding all else constant, when the purchase was made with a `repeat_retailer` we expect an increase of __0.97__ in likelihood that the purchase was __legit__.
- Holding all else constant, when the purchase was made with a `used_pin_number` we expect an increase of __4.38__ in likelihood that the purchase was __legit__.
- Holding all else constant, when the purchase was made with a `online_order` we expect an increase of __3.34__ in likelihood that the purchase was __fraudulent__.

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier()
decision_tree_model.fit(X_train, y_train)
y_preds = decision_tree_model.predict(X_test)

In [ ]:
for pred, actual in zip(y_test[:20], y_preds[:20]):
    print(f'{actual} : {pred}')

In [ ]:
print(f'Accuracy: {decision_tree_model.score(X_test, y_test)}')
print(f'Precision: {precision_score(y_test, y_preds)}')
print(f'F1 score: {f1_score(y_test, y_preds)}')
print(f'Recall: {recall_score(y_test, y_preds)}')
print(f'Confusion Matrix: \n{confusion_matrix(y_test, y_preds)}')

### XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier()
xgb_model.fit(X_train, y_train)
y_preds = xgb_model.predict(X_test)

In [ ]:
print(f'Accuracy: {xgb_model.score(X_test, y_test)}')
print(f'Precision: {precision_score(y_test, y_preds)}')
print(f'F1 score: {f1_score(y_test, y_preds)}')
print(f'Recall: {recall_score(y_test, y_preds)}')
print(f'Confusion Matrix: \n{confusion_matrix(y_test, y_preds)}')

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Get the predicted probabilities for each model
logit_probs = logit_model.predict_proba(X_test)[:, 1]
dt_probs = decision_tree_model.predict_proba(X_test)[:, 1]
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

# Compute the false positive rate, true positive rate, and thresholds for each model
logit_fpr, logit_tpr, _ = roc_curve(y_test, logit_probs)
dt_fpr, dt_tpr, _ = roc_curve(y_test, dt_probs)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)

# Compute the AUC score for each model
logit_auc = roc_auc_score(y_test, logit_probs)
dt_auc = roc_auc_score(y_test, dt_probs)
xgb_auc = roc_auc_score(y_test, xgb_probs)

# Plot the ROC curves
plt.plot(logit_fpr, logit_tpr, label=f'Logistic Regression (AUC = {logit_auc:.2f})')
plt.plot(dt_fpr, dt_tpr, label=f'Decision Tree (AUC = {dt_auc:.2f})')
plt.plot(xgb_fpr, xgb_tpr, label=f'XGBoost (AUC = {xgb_auc:.2f})')

# Plot the diagonal line
plt.plot([0, 1], [0, 1], 'k--')

# Set the labels and title
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()

# Show the plotA
plt.show()

## Using p-values to determine which variables to keep

In [ ]:
import statsmodels.api as sm
import numpy as np

# Fit logistic regression model
model = sm.Logit(y_train, X_train)
result = model.fit()

# Get summary statistics
print(result.summary())

Just kidding... I guess all the features are statistically significant

### SVM using best features

In [ ]:
from sklearn.svm import SVC

# Create an SVM classifier
svm_model = SVC()

X_train = X_train[['distance_from_home', 'repeat_retailer', 'ratio_to_median_purchase_price', 'used_pin_number']]

# Fit the model on the training data
svm_model.fit(X_train, y_train)

# Make predictions on the test data
y_preds = svm_model.predict(X_test)

# Evaluate the model
accuracy = svm_model.score(X_test, y_test)
precision = precision_score(y_test, y_preds)
f1 = f1_score(y_test, y_preds)
recall = recall_score(y_test, y_preds)
confusion = confusion_matrix(y_test, y_preds)

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'F1 score: {f1}')
print(f'Recall: {recall}')
print(f'Confusion Matrix: \n{confusion}')

Took too long (~40 min)